# PharmaShield GCC — Layer 1: Multi-Source Data Ingestion
Group 3 · AI in Operations. Runs plan §5.2 steps 1–3 end to end and writes `data/processed/suppliers_matrix.csv` in the locked schema (§5.1).

Run all cells top to bottom (plan step 10). GenAI disclosure: this notebook and the `layer1_ingestion` package were generated with Claude; prompts are in `docs/GENAI_PROMPT_LOG_layer1.md`.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "layer1_ingestion").exists())
sys.path.insert(0, str(ROOT / "layer1_ingestion"))
import run_ingestion
from ingestion.config import SETTINGS
P = SETTINGS.paths
pd.set_option("display.max_columns", 60, "display.width", 220)
print("SKU:", SETTINGS.sku.sku_id, "| tier", SETTINGS.sku.criticality_tier, "| as of", SETTINGS.as_of)

## Run the pipeline
Live openFDA + World Bank + FDA inspections. Step 1 prints shape and class balance at every step. Set `OFFLINE = True` to replay cached API responses.

In [ ]:
OFFLINE = False
run_ingestion.main(["--log-level", "WARNING"] + (["--offline"] if OFFLINE else []))

## Step 1 — Clean SCMS training data
`late = delivered > scheduled`; dates parsed with `errors='coerce'`; non-numeric freight coerced to NaN.

In [ ]:
clean = pd.read_csv(P.interim / "scms_shipments_clean.csv", low_memory=False)
print("shape:", clean.shape)
print("class balance:", clean["late"].value_counts().to_dict(), f"| late rate {clean['late'].mean():.2%}")
print(clean.groupby("time_split")["late"].agg(["size", "mean"]).round(4))
print("freight status:", clean["freight_status"].value_counts().to_dict())
clean[["id", "scheduled_delivery_date", "delivered_to_client_date", "delay_days", "late", "time_split", "shipment_mode",
       "manufacturing_country", "vendor_inco_term", "freight_cost_usd_coerced", "freight_usd", "weight_kg",
       "freight_to_value_ratio", "unit_price", "line_item_value"]].head()

## Step 2 — Site & corridor features (train period only, no leakage)

In [ ]:
site = pd.read_csv(P.interim / "scms_site_features_train.csv")
corridor = pd.read_csv(P.interim / "scms_corridor_features_train.csv")
display(site.sort_values("site_n_shipments", ascending=False).head(10))
display(corridor.sort_values("corridor_n_shipments", ascending=False).head(10))

## Step 3 — 2026 oncology scoring panel (live openFDA) and entity resolution

In [ ]:
labelers = pd.read_csv(P.interim / "fda_ndc_qualified_labelers.csv")
print(len(labelers), "labelers:", labelers["authorization_type"].value_counts().to_dict())
display(labelers[["labeler_name_fda", "authorization_type", "n_products", "strengths"]])
display(pd.read_csv(P.interim / "plan_vs_live_reconciliation.csv"))
display(pd.read_csv(P.interim / "entity_match_top20_manual_check.csv"))

## Locked schema check — `data/processed/suppliers_matrix.csv` (plan §5.1)

In [ ]:
schema = pd.read_csv(P.reference / "locked_schema.csv")
matrix = pd.read_csv(P.processed / "suppliers_matrix.csv")
assert list(matrix.columns[:16]) == schema["column"].tolist(), "locked column names / order violated"
for c in ["C1_landed_cost", "C5_route_exposure", "hormuz_air_share"]:
    assert matrix[c].between(0, 1).all(), c
assert matrix["manufacturer_id"].is_unique and matrix["criticality_tier"].isin([1, 2, 3]).all()
matrix["shap_top3_reasons"].map(json.loads)
print("Schema OK:", matrix.shape[0], "manufacturers x 16 locked columns (+", matrix.shape[1] - 16, "supporting)")
print("Pending Layer 3:", [c for c in schema["column"] if matrix[c].isna().all()])
matrix.iloc[:, :16]

## Inputs handed to the ML / NLP engines

In [ ]:
display(pd.read_csv(P.interim / "panel_inference_features.csv"))      # C2 feature-based inference (plan 3.1)
display(pd.read_csv(P.processed / "quotes_2026_carboplatin.csv"))     # C1 (plan 4.3)
for name in ["warning_letters_corpus", "contracts_corpus", "fda_recalls_5yr", "fda_inspections_drugs_5yr",
             "panel_facility_inspections", "fda_shortages_sku", "wb_country_indicators"]:
    f = P.interim / f"{name}.csv"
    print(f"{name:32s}", len(pd.read_csv(f, low_memory=False)) if f.stat().st_size > 1 else 0, "rows")

## Data-quality report

In [ ]:
rep = json.loads((P.quality / "quality_report.json").read_text())
display(pd.DataFrame([{"source": k, **v} for k, v in rep["sources"].items()]))
chk = pd.DataFrame(rep["checks"])
chk[chk["severity"].isin(["error", "warning"])][["source", "check", "n_affected", "n_total", "severity", "action", "detail"]]